# Hands-On Evaluation Harness (Building an End-to-End Agent Eval Harness with Phoenix / LangSmith Concepts)
Now that you understand the theory behind trajectory evaluation, tool schema checking, and LLM-as-a-Judge patterns, it is time to tie everything together into a comprehensive, production-grade Evaluation Harness.

In this topic, we build a local Python evaluation harness that loads a golden dataset of test cases, runs an agent against them, records execution traces, grades both intermediate steps and final outputs, and generates a summary report (mirroring core concepts found in observability platforms like Arize Phoenix or LangSmith)


# 1. Architecture of a Custom Eval Harness
A complete agent evaluation harness consists of four modular components:

The Golden Dataset: A list of test cases, each containing a user prompt, expected tool sequences, and expected output criteria.

The Execution Engine: The agent loop that records its trace (thoughts, tool calls, arguments, and final outputs) into a structured log.

The Graders: A suite of test functions combining deterministic checks (tool sequence, argument schema) and semantic checks (LLM-as-a-Judge).

The Reporting Dashboard / CLI Summary: Aggregating scores across the entire dataset to calculate pass rates, average scores, and failure breakdowns.

# 2. Complete Python Implementation
Below is a fully runnable Python script that implements an end-to-end agent evaluation harness.

In [ ]:
import json
from typing import List, Dict, Any
from pydantic import BaseModel, Field

# ==========================================
# 1. DEFINE THE GOLDEN DATASET (Test Cases)
# ==========================================
GOLDEN_DATASET = [
    {
        "test_id": "test_weather_01",
        "user_prompt": "What is the weather in Tokyo?",
        "expected_tools": ["get_weather"],
        "expected_keywords": ["Tokyo", "Celsius"]
    },
    {
        "test_id": "test_math_02",
        "user_prompt": "Calculate 150 multiplied by 4.",
        "expected_tools": ["calculator"],
        "expected_keywords": ["600"]
    }
]

# ==========================================
# 2. SIMULATED AGENT EXECUTION ENGINE
# ==========================================
def run_agent(user_prompt: str) -> Dict[str, Any]:
    """Simulates an agent executing a task and returning a trace and output."""
    trace = []
    
    if "weather" in user_prompt.lower():
        trace.append({
            "step": 1,
            "tool_called": "get_weather",
            "arguments": {"location": "Tokyo"}
        })
        final_output = "The current weather in Tokyo is 22°C and cloudy."
    elif "calculate" in user_prompt.lower():
        trace.append({
            "step": 1,
            "tool_called": "calculator",
            "arguments": {"expression": "150 * 4"}
        })
        final_output = "The result of 150 multiplied by 4 is 600."
    else:
        final_output = "I don't know how to help with that."

    return {
        "trajectory": trace,
        "final_output": final_output
    }

# ==========================================
# 3. EVALUATION GRADERS
# ==========================================
class TestResult(BaseModel):
    test_id: str
    passed: bool
    tool_score: float
    output_score: float
    rationale: str

def evaluate_test_case(test_case: dict, agent_result: dict) -> TestResult:
    # A. Evaluate Tool Selection Trajectory
    expected_tools = test_case["expected_tools"]
    executed_tools = [step["tool_called"] for step in agent_result["trajectory"]]
    tool_match = (executed_tools == expected_tools)
    tool_score = 1.0 if tool_match else 0.0
    
    # B. Evaluate Final Output Keywords (Deterministic Semantic Check)
    final_output = agent_result["final_output"]
    keywords_matched = all(kw.lower() in final_output.lower() for kw in test_case["expected_keywords"])
    output_score = 1.0 if keywords_matched else 0.0
    
    # C. Overall Pass/Fail Gate
    passed = (tool_score == 1.0 and output_score == 1.0)
    
    rationale = f"Tools executed: {executed_tools} (Expected: {expected_tools}). Keywords matched: {keywords_matched}."
    
    return TestResult(
        test_id=test_case["test_id"],
        passed=passed,
        tool_score=tool_score,
        output_score=output_score,
        rationale=rationale
    )

# ==========================================
# 4. RUNNER & REPORT AGGREGATOR
# ==========================================
def run_evaluation_harness(dataset: List[dict]) -> None:
    print("==================================================")
    print("🚀 STARTING AGENT EVALUATION SUITE RUN...")
    print("==================================================\n")
    
    results: List[TestResult] = []
    
    for test_case in dataset:
        print(f"Running Test ID: {test_case['test_id']}...")
        
        # Run agent
        agent_output = run_agent(test_case["user_prompt"])
        
        # Evaluate run
        result = evaluate_test_case(test_case, agent_output)
        results.append(result)
        
        status_icon = "✅ PASS" if result.passed else "❌ FAIL"
        print(f" -> {status_icon} | Tool Score: {result.tool_score} | Output Score: {result.output_score}")
        print(f"    Rationale: {result.rationale}\n")

    # Aggregate Metrics
    total_tests = len(results)
    passed_tests = sum(1 for r in results if r.passed)
    pass_rate = (passed_tests / total_tests) * 100
    
    print("==================================================")
    print("📊 EVALUATION SUITE SUMMARY REPORT")
    print("==================================================")
    print(f"Total Tests Run: {total_tests}")
    print(f"Passed: {passed_tests}")
    print(f"Failed: {total_tests - passed_tests}")
    print(f"Overall Success Rate: {pass_rate:.1f}%")
    print("==================================================")

if __name__ == "__main__":
    run_evaluation_harness(GOLDEN_DATASET)

# 3. Key Takeaways
Modular Separation: Show students how keeping the dataset, agent runner, and evaluation graders cleanly separated allows them to swap out agent versions or models without rewriting their test harness.

Bridge to Production Tools: Explain that while custom harnesses are great for learning, enterprise teams plug this exact loop into platforms like Arize Phoenix, LangSmith, or Braintrust to log traces automatically to the cloud and visualize failing steps in real-time UIs.